<a href="https://colab.research.google.com/github/agil-p-varghese/Fine-Tunning--Models/blob/main/RAG_learning_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install chromadb huggingface_hub

In [ ]:
import os
from google.colab import userdata
import chromadb
from huggingface_hub import InferenceClient

In [ ]:
#load token from secrets
os.environ["HF_TOKEN"]=userdata.get('HF_TOKEN')

In [ ]:
#initiate hugging face inference client
client=InferenceClient(token=os.environ["HF_TOKEN"])
chat_model="meta-llama/Meta-Llama-3-8B-Instruct"

In [ ]:
#initialize light weight chroma db
chroma_client=chromadb.PersistentClient(path="./chroma_db_cloud")
collection=chroma_client.get_or_create_collection(name="cloud_learning_rag")

In [ ]:
#sample knowledge base
documents=[
    "Agil is a associate software engineer working currently in Accenture Mumbai.His home place is Kerala,Trissur,Chalakudy,Padinjakkara House.",
    "applixe is a fruit seen in only in india,kerala,periyar.it is very healthy.it can cure stage-2 cancer(blood cancer).",
    "ZimbaLLM is a AI startup headquatered in india.it conducts research in advanced super intelligence (SI).",
]

print("Ingesting documents into ChromaDB .....")
for i,doc in enumerate(documents):
  collection.upsert(
    ids=[str(i)],
    documents=[doc]
  )


In [ ]:
#ask a question
user_question="who is Agil?where is he now?what is his profession?"
print(f"\nUser Question :{user_question}")


In [ ]:
#retrive top relevant chunks from ChromaDB
  results=collection.query(
      query_texts=[user_question],
      n_results=2
  )
  retrieved_chunks=results["documents"][0]
  context="\n\n".join(retrieved_chunks)

  print("\n---------Retrieved Context---------\n")
  print(context)
  print("-------------------------------------")


In [ ]:
#Generate Grounded Response Via Hugging Face Cloud Inferece API
messages=[
    {
        "role":"system",
        "content":"You are helpfull assistant.answer the question only using the provided context.if answer is not in context ,say i dont know."
    },
    {
      "role":"user",
      "content":f"Context: \n{context} \n\n Question :{user_question}"
    }
]
print("Calling Hugging face Cloud API for generation.....")
response=client.chat_completion(
  model=chat_model,
  messages=messages,
  max_tokens=250,
  temperature=0.1
)
print("\n Final Answer:")
print(response.choices[0].message.content)